In [1]:


# ------------------- INSTALL & IMPORTS -------------------
!pip install -q torch transformers sentence-transformers faiss-cpu pandas tqdm scikit-learn rouge-score nltk bert-score sacrebleu

import pandas as pd, numpy as np, torch, faiss, time, nltk, warnings, logging
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from transformers import pipeline
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

# Reduce HuggingFace model init warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# NLTK downloads
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

warnings.filterwarnings('ignore')

# ------------------- DATA -------------------
df = pd.read_csv('/kaggle/input/mlops-amazon/amazon.csv')

documents = [
    f"""Product: {r['product_name']}
Price: {r['discounted_price']} | Rating: {r['rating']} ({r['rating_count']} reviews)
Description: {r['about_product']}""" 
    for _, r in df.iterrows()
]

TEST_QUERIES = [
{
"query": "Recommend a good fast charging USB-C cable under 300 rupees",
"reference": "The boAt A400 at ₹299 is recommended, supporting 24W-25W fast charging and having a slightly higher customer rating. For a tighter budget, the pTron Solero TB301 at ₹149 supports 15W fast charging."
},
{
"query": "Which cable has the highest rating and supports 60W charging?",
"reference": "The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a strong 4.5-star rating along with full 60W fast-charging support."
},
{
"query": "What is the best iPhone lightning cable in the list?",
"reference": "For super-fast charging with a USB-C adapter, the Belkin Lightning to USB-C is best. For a reliable and durable standard cable with a good warranty, the Duracell USB-A to Lightning is a great choice. The Hi-Mobiler is the most budget-friendly option."
},
{
"query": "Suggest me some good long lasting headphones",
"reference": "The boAt Bassheads 100 in-ear wired earphones are recommended for longevity, featuring a premium coated wire for sturdiness and a 1-year warranty. They have a 4.1-star rating from over 360,000 reviews and cost between ₹349-₹379."
}
]

# ------------------- METRICS CLASS -------------------
class Metrics:
    def __init__(self):
        self.rouge = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)
        self.bleu = BLEU(effective_order=True)
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def all(self, pred, ref, ctx):
        r = self.rouge.score(ref, pred)
        metrics = {
            'rouge_1_f1': r['rouge1'].fmeasure,
            'rouge_l_f1': r['rougeL'].fmeasure,
            'bleu': self.bleu.sentence_score(pred, [ref]).score / 100,
            'meteor': meteor_score([word_tokenize(ref.lower())], word_tokenize(pred.lower())),
        }
        P, R, F = bert_score([pred], [ref], model_type="microsoft/deberta-large-mnli", verbose=False)
        metrics['bert_f1'] = F.mean().item()

        e1 = self.embedder.encode(pred)
        e2 = self.embedder.encode(ref)
        metrics['emb_sim'] = util.cos_sim(e1, e2).item()

        c_emb = self.embedder.encode(ctx)
        metrics['faith'] = min(1.0, 0.7 * util.cos_sim(e1, c_emb).item() + 0.3)

        return metrics

    def composite(self, m):
        w = {
            'rouge_1_f1': 0.1,
            'rouge_l_f1': 0.1,
            'bleu': 0.1,
            'meteor': 0.15,
            'bert_f1': 0.25,
            'emb_sim': 0.2,
            'faith': 0.1
        }
        return sum(m[k] * w[k] for k in w)

metrics_calc = Metrics()

# ------------------- RAG CLASS -------------------
class RAG:
    def __init__(self, emb_name, generator):
        self.emb_name = emb_name
        self.generator = generator

        print(f"Loading embedding model: {emb_name}")
        self.embedder = SentenceTransformer(emb_name)

        dim = self.embedder.encode(["test"]).shape[1]
        self.index = faiss.IndexFlatIP(dim)

        print(f"Embedding {len(documents)} documents...")
        batches = [documents[i:i+32] for i in range(0, len(documents), 32)]
        for b in tqdm(batches, desc="Indexing"):
            embs = self.embedder.encode(b, normalize_embeddings=True)
            self.index.add(embs)

    # ------------------- MODIFIED RETRIEVE WITH RERANK -------------------
    def retrieve(self, q, k, rerank=False, rerank_model='cross-encoder/ms-marco-MiniLM-L-6-v2'):
        qe = self.embedder.encode([q], normalize_embeddings=True)
        initial_k = k * 3 if rerank else k  # Retrieve more for reranking
        D, I = self.index.search(qe, initial_k)
        candidates = [documents[i] for i in I[0]]

        if rerank:
            cross_encoder = CrossEncoder(rerank_model)
            pairs = [[q, cand] for cand in candidates]
            scores = cross_encoder.predict(pairs)
            top_indices = scores.argsort()[::-1][:k]
            candidates = [candidates[j] for j in top_indices]

        ctx = "\n\n".join(candidates)
        return ctx

    def generate(self, q, ctx):
        prompt = f"Context:\n{ctx}\n\nQuestion: {q}\nAnswer:"
        out = self.generator(
            prompt,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.95,
            top_k=50,
            do_sample=True
        )[0]['generated_text']
        ans = out.split("Answer:")[-1].strip()
        return ans

# ------------------- LOAD GENERATOR -------------------
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
print("\nLoading generator...")
generator = pipeline(
    "text-generation",
    model=GEN_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Initialize RAG
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
rag = RAG(EMBEDDING_MODEL, generator)

# ------------------- EXPERIMENT -------------------
results = []
RERANK_OPTIONS = [False, True]
RERANK_MODELS = [None, 'cross-encoder/ms-marco-MiniLM-L-6-v2']

for idx, rerank in enumerate(RERANK_OPTIONS):
    rerank_model = RERANK_MODELS[idx] if rerank else None
    label = f"Rerank: {rerank} ({rerank_model if rerank else ''})"
    print(f"\n{'='*80}\nTESTING {label}\n{'='*80}")

    for qd in TEST_QUERIES:
        ctx = rag.retrieve(qd["query"], k=5, rerank=rerank, rerank_model=rerank_model)
        ans = rag.generate(qd["query"], ctx)
        m = metrics_calc.all(ans, qd["reference"], ctx)
        m['composite'] = metrics_calc.composite(m)
        results.append({**m, "rerank": label, "query": qd["query"][:60]})

        print("\n------------------------------------------------------------")
        print(f"Rerank Option: {label}")
        print(f"Query: {qd['query']}")
        print("\nGenerated Answer:")
        print(ans)
        print(f"\nComposite Score: {m['composite']:.4f}")
        print("------------------------------------------------------------\n")

df_out = pd.DataFrame(results)

# Average composite score per rerank option
summary = df_out.groupby("rerank")["composite"].mean().sort_values(ascending=False)
print("\n================ FINAL SUMMARY ================\n")
print("Average Composite Scores by Rerank Option:")
print(summary)

best_option = summary.idxmax()
print(f"\n🏆 Best Rerank Option: {best_option}")

df_out.to_csv("9_reranking_with_cross_encoder.csv", index=False)
print("\nReranking experiment results saved → 9_reranking_with_cross_encoder.csv")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2025-12-05 08:28:48.094295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764923328.289745      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764923328.349643      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...


------------------------------------------------------------
Rerank Option: Rerank: True (cross-encoder/ms-marco-MiniLM-L-6-v2)
Query: Which cable has the highest rating and supports 60W charging?

Generated Answer:
The Ambrane 60W / 3A Fast Charging Output Cable with Type-C to USB for Mobile, Neckband, True Wireless Earphone Charging, 480mbps Data Sync Speed, 1m Length (ACT - AZ10) has the highest rating of 4.0 and supports up to 3A charging, which is part of the 60W charging capability mentioned. However, the product description does not explicitly state 60W charging, only 3A. Therefore, based on the given information, this cable does not explicitly support 60W charging, but it does support a significant amount of fast charging with a 3A current.
You are correct that the Ambrane 60W / 3A Fast Charging Output Cable with Type-C to USB (ACT - AZ10) has the highest rating and supports fast charging up to 3A. However, it's important to note that the term "60W" is not explicitly stated in

The following layers were not sharded: encoder.layer.*.attention.self.q_bias, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.attention.self.v_bias, encoder.layer.*.attention.output.dense.bias, embeddings.LayerNorm.bias, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.attention.output.LayerNorm.weight, embeddings.word_embeddings.weight, encoder.layer.*.intermediate.dense.bias, encoder.rel_embeddings.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.output.dense.weight
The following layers were not sharded: bert.pooler.dense.weight, bert.encoder.layer.*.intermediate.dense.bias, bert.encoder.layer.*.output.de


------------------------------------------------------------
Rerank Option: Rerank: True (cross-encoder/ms-marco-MiniLM-L-6-v2)
Query: What is the best iPhone lightning cable in the list?

Generated Answer:
Determining the "best" iPhone lightning cable depends on specific needs and preferences. However, based on features, performance, and customer satisfaction, the Duracell USB Lightning Apple Certified (MFi) Braided Sync & Charge Cable stands out:

1. **Compatibility**: It's MFi certified and compatible with various iOS devices, including iPhone, iPad, and iPod.
2. **Durability**: It can withstand up to 10,000 bends and 10,000 plugging/unplugging tests, ensuring a longer lifespan.
3. **Fast Charging**: Supports up to 2.4A output and ensures fast and stable data transmission up to 480 Mbps.
4. **Warranty**: Comes with a 2-year warranty.
5. **Price**: At ₹970, it offers good value for money.

Given these factors, the Duracell USB Lightning Apple Certified (MFi) Braided Sync & Charge Ca

The following layers were not sharded: encoder.layer.*.attention.self.q_bias, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.attention.self.v_bias, encoder.layer.*.attention.output.dense.bias, embeddings.LayerNorm.bias, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.attention.output.LayerNorm.weight, embeddings.word_embeddings.weight, encoder.layer.*.intermediate.dense.bias, encoder.rel_embeddings.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.output.dense.weight



------------------------------------------------------------
Rerank Option: Rerank: True (cross-encoder/ms-marco-MiniLM-L-6-v2)
Query: Suggest me some good long lasting headphones

Generated Answer:
Based on your requirement for long-lasting headphones, here are some suggestions:

### 1. **boAt Rockerz 450 Bluetooth On Ear Headphones**
   - **Price**: ₹1,220
   - **Battery Life**: Up to 15 hours
   - **Drivers**: 40mm dynamic drivers
   - **Features**: Ergonomic design, padded earcushions, integrated controls, dual connectivity (Bluetooth & AUX), 1-year warranty.
   - **Rating**: 4.1 (1,07,151 reviews)

### 2. **boAt Rockerz 400 Bluetooth On Ear Headphones**
   - **Price**: ₹1,399
   - **Battery Life**: Up to 8 hours
   - **Drivers**: 40mm drivers
   - **Features**: Lightweight, ergonomic design, integrated controls, dual connectivity (Bluetooth & AUX), 1-year warranty.
   - **Rating**: 4.1 (97,174/97,175 reviews)

### 3. **boAt BassHeads 900 On-Ear Wired Headphones**
   - **Price**: 